# Chapter 9: Tree Ensembles — Random Forest & Gradient Boosting
### Comparative Performance on Credit Risk Underwriting Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/src/ch09_trees_and_ensembles.ipynb)

**Modern Machine Learning & Data Science Engineering**  
Author: *Vaibhav Rastogi*


### 0. Environment Setup & Cloud Dataset Download

In [ ]:
import os, urllib.request
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, 'credit_risk_dataset.csv')
if not os.path.exists(csv_path):
    url = 'https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/credit_risk_dataset.csv'
    urllib.request.urlretrieve(url, csv_path)
print(f'Dataset ready at {csv_path}')


### 1. Categorical Encoding & Dataset Partitioning

In [ ]:
df = pd.read_csv(csv_path)
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df_encoded.drop(columns=['loan_status'])
y = df_encoded['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f'Feature matrix dimensions: {X_train.shape}')


### 2. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
print(f'Random Forest ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}')


### 3. Gradient Boosting (LightGBM)

In [ ]:
try:
    import lightgbm as lgb
    lgb_model = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)
    lgb_probs = lgb_model.predict_proba(X_test)[:, 1]
    print(f'LightGBM Classifier ROC-AUC: {roc_auc_score(y_test, lgb_probs):.4f}')
except ImportError:
    print('LightGBM not installed. Run: !pip install lightgbm')
